In [ ]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath(".."))
import config

In [ ]:
config.path_to_evaluation

In [ ]:
metrics = ["NRMSE", "MASE", "sMAPE"]
p_list = ["5000", "12000", "25000", "50000", "100000", "all"]
feats = ['abs_energy','intermittency','mean','median','kurtosis','skewness','standard_deviation','agg_autocorrelation_max','erraticness','agg_linear_trend_slope']
runs =  ["case1_run1", "case1_run2", "case1_run3", "case1_run4", "case1_run5"]

Calculate FID

In [ ]:
from scipy.linalg import sqrtm
def calculate_fid(
        source_feat: np.ndarray = None,
        target_feat: np.ndarray = None,
    ) -> float:

        mean_real = np.mean(source_feat, axis=0)
        cov_real = np.cov(source_feat, rowvar=False)

        mean_gen = np.mean(target_feat, axis=0)
        cov_gen = np.cov(target_feat, rowvar=False)

        diff_means = np.sum(np.square(mean_real - mean_gen))
        cov_prod = sqrtm(cov_real.dot(cov_gen))

        if np.iscomplexobj(cov_prod):
            cov_prod = cov_prod.real

        fid = diff_means + np.trace(cov_real + cov_gen - 2.0 * cov_prod)

        return fid

In [ ]:
results = []

for run in runs: 
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"
    for div in [10, 5, 0]:
        targets_feat = []
        targets_ids =[]
        for p in p_list:
            concat = pd.read_parquet(f"{config.path_to_data}/concat/CONCAT_feat.parquet")
            target = pd.read_parquet(f"{config.path_to_data}/{run}/interim/target_div{div}_sim{p}{run}/target_div{div}_sim{p}{run}.parquet")
            target_ids = target.id_ts.unique().tolist()
            target_feat = concat[concat["id_ts"].isin(target_ids)][feats + ["id_ts"]]
            targets_feat.append(target_feat)
            targets_ids.append(target_ids)
        targets_feat = pd.concat(targets_feat)
        source = pd.read_parquet(f"{config.path_to_data}/{run}/interim/source_div{div}{run}/source_div{div}{run}.parquet")
        source_ids = source.id_ts.unique().tolist()
        source_feat = concat[concat["id_ts"].isin(source_ids)][feats + ["id_ts"]]
        all_feat = pd.concat([targets_feat, source_feat], axis=0)
        feat_std = all_feat.copy()
        feat_std[feats] = (all_feat[feats] - all_feat[feats].mean()) / all_feat[feats].std()
        for i,p in zip(range(len(p_list)), p_list):
            target_feat_std = feat_std[feat_std["id_ts"].isin(targets_ids[i])]
            source_feat_std = feat_std[feat_std["id_ts"].isin(source_ids)]
            fid = calculate_fid(target_feat_std.drop(columns=["id_ts"]).to_numpy(), source_feat_std.drop(columns=["id_ts"]).to_numpy())
            results.append({
                "run": run,
                "div": div,
                "p": p,
                "fid": fid
            })
fid = pd.DataFrame(results)
fid.to_parquet(f"{config.path_to_evaluation}/fid_similarity.parquet")
        